In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

bronze_path = f"abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze"

print("Bronze path:", bronze_path)

Bronze path: abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze


In [0]:
%sql
use catalog pharma_catalog;
use schema bronze;

In [0]:
file_table_map = {
    "Employee_Master_RAW.csv": "employee_master",
    "Customer_HCP_Master_RAW.csv": "customer_hcp_master",
    "Employee_Product_Assignment_RAW.csv": "employee_product_assignment",
    "Monthly_Targets_RAW.csv": "monthly_targets",
    "Product_Master_RAW.csv": "product_master",
    "Sales_Transactions_RAW.csv": "sales_transactions"
}

In [0]:
%python
#reading csv
def read_csv(file_path):

    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path)

    return df

In [0]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp
for file_name, table_name in file_table_map.items():

    file_path = f"{bronze_path}/{file_name}"

    print(f"Processing file: {file_name}")

    df = read_csv(file_path)

    # Add metadata columns
    df = df.withColumn("insert_timestamp", from_utc_timestamp(current_timestamp(), "Asia/Kolkata")) \
           .withColumn("source_file", input_file_name())

    # Write as Delta table in Unity Catalog
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"pharma_catalog.bronze.{table_name}")

    print(f"Loaded table: pharma_catalog.bronze.{table_name}")

Processing file: Employee_Master_RAW.csv
Loaded table: pharma_catalog.bronze.employee_master
Processing file: Customer_HCP_Master_RAW.csv
Loaded table: pharma_catalog.bronze.customer_hcp_master
Processing file: Employee_Product_Assignment_RAW.csv
Loaded table: pharma_catalog.bronze.employee_product_assignment
Processing file: Monthly_Targets_RAW.csv
Loaded table: pharma_catalog.bronze.monthly_targets
Processing file: Product_Master_RAW.csv
Loaded table: pharma_catalog.bronze.product_master
Processing file: Sales_Transactions_RAW.csv
Loaded table: pharma_catalog.bronze.sales_transactions


In [0]:
%sql
show tables in pharma_catalog.bronze;

database,tableName,isTemporary
bronze,customer_hcp_master,false
bronze,employee_master,false
bronze,employee_product_assignment,false
bronze,monthly_targets,false
bronze,product_master,false
bronze,sales_transactions,false


In [0]:
%sql
select * 
from pharma_catalog.bronze.employee_master
limit 10;

employee_id,employee_code,first_name,last_name,gender,date_of_birth,join_date,department,designation,manager_id,zone,region,state,city,hq_code,email,phone,employment_status,base_salary,incentive_plan,source_system,created_at,updated_at,batch_id,extra_notes,insert_timestamp,source_file
E100001,EMP1001,Riya,Gupta,Male,19-06-1999,2022-08-03,Sales,Product Executive,null,North,Delhi,Delhi,Gurugram,HQ-N-127,riya.gupta1@pharmaco.com,+91-9012167820,Active,"7,42,297.00",Plan B,Legacy ERP,2024-01-07T05:44:27Z,2025-06-24T08:09:13Z,BATCH_20260313_002,null,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100002,EMP1002,Priya,Agarwal,Female,1995-08-11,2023-03-08,Marketing,Territory Business Executive,E100001,North,Uttar Pradesh,Uttar Pradesh,Lucknow,HQ-N-146,priya.agarwal2@pharmaco.com,+91-6342703921,Active,"4,91,815.00",Plan B,Legacy ERP,2025-04-09T20:53:23Z,2025-03-25T11:22:13Z,BATCH_20260313_003,Promoted,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100003,EMP1003,Ananya,Joshi,Female,1987-02-09,2019-04-04,Medical Affairs,Medical Representative,null,North,Rajasthan,Rajasthan,Udaipur,HQ-N-334,ananya.joshi3@pharmaco.com,+91-7722989659,Active,"7,22,463.00",Plan C,Legacy ERP,2024-08-05T20:31:25Z,2025-11-26T14:09:16Z,BATCH_20260313_001,Transferred,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100004,EMP1004,Swati,Kapoor,Male,1981-03-16,2017-06-17,Finance,Medical Representative,E100001,Central,Rajasthan,Rajasthan,Jodhpur,HQ-C-532,swati.kapoor4@pharmaco.com,+91-8559321289,On Leave,"13,89,633.00",Plan B,SAP,2025-11-27T23:07:43Z,2025-10-02T08:49:41Z,BATCH_20260313_002,null,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100005,EMP1005,Vikram,Sharma,Male,15-06-1997,2017-12-14,HR,Territory Business Executive,E100001,East,Jharkhand,Jharkhand,Ranchi,HQ-E-897,vikram.sharma5@pharmaco.com,+91-8572447412,Active,"3,20,845.00",Plan A,Legacy ERP,2024-11-10T07:03:15Z,2025-10-18T02:05:46Z,BATCH_20260313_002,null,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100006,EMP1006,Naveen,Joshi,Male,1993-12-24,2021-10-20,Sales,Area Sales Manager,null,Central,Chhattisgarh,Chhattisgarh,Raipur,HQ-C-787,naveen.joshi6@pharmaco.com,+91-8222971316,Active,"7,99,895.00",Plan A,SAP,2024-12-12T00:37:35Z,2025-04-28T18:14:00Z,BATCH_20260313_001,null,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100007,EMP1007,Neha,Patel,Female,1984-08-11,2020-06-11,Sales,Medical Representative,E100006,South,Karnataka,Karnataka,Bengaluru,HQ-S-516,neha.patel7@pharmaco.com,+91-7851350739,Active,"11,42,143.00",Plan B,SAP,2025-11-20T20:41:06Z,2025-02-01T12:46:21Z,BATCH_20260313_001,Transferred,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100008,EMP1008,Nisha,Nair,Male,1978-05-20,2017-10-20,Sales,Medical Representative,E100007,South,Tamil Nadu,Tamil Nadu,Madurai,HQ-S-969,nisha.nair8@pharmaco.com,+91-8067418686,Active,"11,21,042.00",Plan A,CRM,2025-01-23T00:24:16Z,2026-02-06T14:18:27Z,BATCH_20260313_003,Backfill,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100009,EMP1009,KIara,Singh,Male,1977/09/25,29-01-2021,HR,Area Sales Manager,E100001,West,Gujarat,Gujarat,Ahmedabad,HQ-W-170,kiara.singh9@pharmaco.com,+91-7010193046,Active,"14,74,695.00",Plan A,SAP,2025-09-26T02:26:42Z,2025-10-26T18:33:20Z,BATCH_20260313_002,Transferred,2026-03-16T14:37:06.476722Z,abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/bronze/Employee_Master_RAW.csv
E100010,EMP1010,karan,Nair,Male,1978-04-03,2018-07-19,Medical Affairs,Medical Representative,E100002,East,Odisha,Odisha,Cuttack